# Chapter 5 — Building the original implementation

The reproduction lives in `research/pel/`, a plain Python package next to (and independent of) the repository's ROS 2 packages. This chapter follows one label through the whole system, reading the real source as it goes.

## 5.1 Architecture

| File | Paper concept | Depends on |
|---|---|---|
| `observation_graph.py` | $G_o$, recognition $f_r$ | — |
| `text_similarity.py` | Levenshtein similarity | — |
| `wordnet_access.py` | `GET_SYNSET`, `GET_NAMES`, `GET_SIMILARITY` | NLTK WordNet 3.0 |
| `knowledge_graphs.py` | the three target graphs, queries S / S_links / L / L_links | — |
| `sparql_client.py` | HTTP + persistent response cache | requests, sqlite |
| `candidate_selection.py` | Algorithm 1 and pooling of results | the two above |
| `disambiguation.py` | Algorithm 2 | WordNet |
| `mapping.py` | $f_q$, $f_c$, $f_c + f_d$ | all of the above |
| `evaluation.py` | hit@k, Wilson interval, sample size | — |
| `recorded_access.py` | offline replay of recorded graph answers (used in this tutorial) | — |

Design rules followed: one paper concept per module; function names mirror the paper's pseudocode; network access is confined to `sparql_client.py` and `mapping.KnowledgeGraphAccess`, so Algorithms 1 and 2 are pure functions that can be unit-tested offline.

In [1]:
# Locate the research/ directory and make the real implementation importable.
import sys, json
from pathlib import Path
RESEARCH = Path.cwd().resolve()
while not (RESEARCH / "pel").is_dir():
    RESEARCH = RESEARCH.parent
for path in (RESEARCH, RESEARCH / "experiments"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
FIGURES = RESEARCH / "experiments" / "figures"
RESULTS = RESEARCH / "experiments" / "results"
%matplotlib inline
import inspect

## 5.2 Step 0 — from a detection to an observation graph

In [2]:
from pel.observation_graph import from_detections
graph = from_detections([("orange", (50, 50, 40, 30), 0.95)], time=62903.0)
label = graph.labels()[0]
graph.triples()

[('entity_1', 'hasLabel', 'orange'),
 ('entity_1', 'hasLocation', (50.0, 50.0, 40.0, 30.0)),
 ('entity_1', 'hasTime', 62903.0),
 ('entity_1', 'hasCertainty', 0.95)]

## 5.3 Step 1 — Algorithm 1 in code

The comments on the right give the line numbers of the paper's pseudocode.

In [3]:
from pel import candidate_selection
print(inspect.getsource(candidate_selection.select_candidate_names))

def select_candidate_names(label: str) -> list[tuple[float, str]]:
    """Algorithm 1 (SELECT_CANDIDATES): alternate names for a perceived label.

    Returns ``[(similarity, name), ...]`` sorted by decreasing similarity
    (ties keep WordNet order).  If WordNet does not know the label, the label
    itself is the only name (Algorithm 1, lines 4-6).  The label is always
    present with similarity 1, because it is a lemma of its own synsets.
    """
    synsets = get_synsets(label)                                   # line 2
    if not synsets:                                                # lines 4-6
        return [(1.0, label)]
    scored_names: dict[str, float] = {}                            # line 3
    for synset in synsets:                                         # line 7
        for name in get_names(synset):                             # lines 8-9
            if name not in scored_names:
                scored_names[name] = levenshtein_similarity(label, name)  # line 10
    

In [4]:
scored_names = candidate_selection.select_candidate_names(label)
scored_names

[(1.0, 'orange'),
 (0.8333333333333334, 'Orange'),
 (0.625, 'orangish'),
 (0.6, 'orangeness'),
 (0.5454545454545454, 'orange tree'),
 (0.41666666666666663, 'Orange River')]

## 5.4 Step 2 — querying the graph

`knowledge_graphs.build_retrieval_query` writes the SPARQL; `KnowledgeGraph.surface_form` applies the graph's casing convention first (interpretation I3):

In [5]:
from pel.knowledge_graphs import WIKIDATA, DBPEDIA, build_retrieval_query
print(WIKIDATA.surface_form("orange"), "|", DBPEDIA.surface_form("orange"))
print(build_retrieval_query(DBPEDIA.surface_form("orange"), "L_links", 50))

orange | Orange
SELECT ?entity (COUNT(?target) AS ?links) WHERE { ?entity <http://www.w3.org/2000/01/rdf-schema#label> "Orange"@en . ?entity ?outgoing ?target } GROUP BY ?entity ORDER BY DESC(?links) LIMIT 50


Live access goes through `mapping.KnowledgeGraphAccess`, whose three methods are all the pipeline needs: `retrieve(name, query_type)`, `class_labels(iris)`, and (for the improved method) `descriptions(iris)`. Every response is cached in SQLite by `sparql_client.CachedSparqlClient`; a query that times out twice is recorded as an empty result, while network errors are never recorded.

In this notebook we use `RecordedAccess`, which implements the same three methods from a recording of the live answers:

In [6]:
from pel.recorded_access import RecordedAccess
wikidata = RecordedAccess("coco", "wikidata")
for score, name in scored_names:
    print(f"{name!r:16} -> {[iri.split('/')[-1] for iri, _ in wikidata.retrieve(name, 'L_links')][:6]}")

'orange'         -> ['Q13191', 'Q39338', 'Q59318527', 'Q3040333', 'Q7099506']
'Orange'         -> ['Q1431486', 'Q187796', 'Q491350', 'Q580973', 'Q117774933', 'Q2028096']
'orangish'       -> []
'orangeness'     -> []
'orange tree'    -> ['Q3621273']
'Orange River'   -> ['Q181475', 'Q2028124', 'Q17183950', 'Q2028128', 'Q7099710', 'Q7099709']


## 5.5 Step 3 — pooling the results (interpretation I1)

The paper says all returned entities “are added as potential candidates” but not in which order. `collect_candidates` implements our reading:

In [7]:
print(inspect.getsource(candidate_selection.collect_candidates).split('"""')[2])


    pooled: list[Candidate] = []
    seen: set[str] = set()
    for similarity, name in scored_names:
        for iri, links in retrieve(name, query_type):
            if iri not in seen:
                seen.add(iri)
                pooled.append(Candidate(iri, name, similarity, links))
    if query_type.endswith("_links"):
        pooled.sort(key=lambda candidate: -(candidate.links or 0))   # stable
    return pooled



In [8]:
from pel.mapping import query_only, candidate_selection as run_candidate_selection
f_q = query_only(label, wikidata, "L_links")
f_c = run_candidate_selection(label, wikidata, "L_links")
show = lambda cs: [(c.iri.split("/")[-1], c.links, c.source_name) for c in cs[:6]]
print("f_q :", show(f_q))
print("f_c :", show(f_c))

f_q : [('Q13191', 586, 'orange'), ('Q39338', 568, 'orange'), ('Q59318527', 95, 'orange'), ('Q3040333', 47, 'orange'), ('Q7099506', 13, 'orange')]
f_c : [('Q13191', 586, 'orange'), ('Q39338', 568, 'orange'), ('Q1431486', 565, 'Orange'), ('Q187796', 498, 'Orange'), ('Q181475', 356, 'Orange River'), ('Q491350', 337, 'Orange')]


Compare the two lists. With the `_links` ordering applied to the *pooled* set, entities found through other names (`Orange`, `Orange River`) are interleaved with those found through the label itself, purely by number of statements. Here the fruit keeps rank 1; for other labels a popular entity reached through a synonym takes it. This is the mechanism behind the paper's remark that candidate selection alone can lower accuracy.

## 5.6 Step 4 — Algorithm 2 in code

In [9]:
from pel import disambiguation
print(inspect.getsource(disambiguation.class_similarity))
print(inspect.getsource(disambiguation.disambiguate).split('"""')[2])

def class_similarity(label_synsets, class_labels: list[str]) -> float:
    """Inner loops of Algorithm 2 (lines 6-13) for one candidate."""
    high_score = 0.0                                               # line 6
    for class_label in class_labels:                               # line 7
        for class_synset in get_synsets(class_label):              # lines 8-9
            for label_synset in label_synsets:
                sim_score = path_similarity(class_synset, label_synset)   # line 10
                high_score = max(sim_score, high_score)            # line 11
    return high_score


    label_synsets = get_synsets(label)                             # line 2
    scores = [(class_similarity(label_synsets, class_labels_of.get(candidate.iri, [])),
               candidate) for candidate in candidates]             # lines 4-15
    scores.sort(key=lambda scored: -scored[0])                     # line 16 (stable)
    return scores                                                  

In [10]:
classes = wikidata.class_labels([c.iri for c in f_c])
for score, candidate in disambiguation.disambiguate(label, f_c, classes)[:6]:
    print(f"{score:.2f}  {candidate.iri.split('/')[-1]:>10}  via {candidate.source_name!r:14} classes: {classes[candidate.iri][:4]}")

0.50      Q13191  via 'orange'       classes: ['hesperidium', 'common name', 'tropical fruit', 'citrus fruit']
0.50     Q181475  via 'Orange River' classes: ['river']
0.50   Q17183950  via 'Orange River' classes: ['river']
0.50    Q2028128  via 'Orange River' classes: ['river']
0.50    Q7099710  via 'Orange River' classes: ['river']
0.50    Q7099709  via 'Orange River' classes: ['river']


## 5.7 The three processes behind one function

`mapping.map_label(label, access, query_type, process)` returns the ranked IRIs for one condition of the paper's Table 1; rank 1 is the link, an empty list is the NIL link.

In [11]:
from pel.mapping import map_label
for example in ("orange", "couch", "dog"):
    for process in ("fq", "fc", "fc+fd"):
        top = map_label(example, wikidata, "L_links", process)[:3]
        described = [wikidata.descriptions([i])[i][:28] or i.split("/")[-1] for i in top]
        print(f"{example:7s} {process:6s} {described}")
    print()

orange  fq     ['citrus fruit of the orange t', 'colour, located between red ', 'eye color']
orange  fc     ['citrus fruit of the orange t', 'colour, located between red ', 'French multinational telecom']
orange  fc+fd  ['citrus fruit of the orange t', 'major river of southern Afri', 'river in Florida, United Sta']

couch   fq     ['piece of furniture for seati', 'long seating furniture prima', 'Q131593976']
couch   fc     ['piece of furniture for seati', 'decorative edging for a pict', 'data transmission unit']


couch   fc+fd  ['decorative edging for a pict', 'piece of furniture for seati', 'long seating furniture prima']

dog     fq     ['domesticated mammal of the f', 'motif', 'command-line DNS client']
dog     fc     ['domesticated mammal of the f', 'sausage in bun, usually with', 'rear appendage of organism']


dog     fc+fd  ['object from Hunterian Art Ga', 'Object from Hunterian Art Ga', 'type of hunting dog used by ']



Two things to notice. For `couch`, disambiguation promotes a *picture frame* — “frame” is a verb synonym of “to couch”. For `dog`, the top results under `fc+fd` are museum objects, not the animal. Chapter 7 explains both.

## 5.8 Testing

Each component has an offline test in `research/tests/`. The tests encode the *definitions* (symmetry and triangle inequality of the edit distance; path similarity equals $1/(1+d)$; hit@k arithmetic; the paper's sample-size figure) and the *behaviour* of the algorithms on small hand-made graphs.

In [12]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", str(RESEARCH / "tests"), "-q"],
                        capture_output=True, text=True, cwd=RESEARCH.parent, env={"PYTHONPATH": str(RESEARCH), "PATH": ""})
print(result.stdout[-400:])

.......................................                                  [100%]
39 passed in 55.05s



| What is tested | How |
|---|---|
| Levenshtein | known distances, symmetry, triangle inequality, bounds of the similarity |
| Algorithm 1 | exact name list for `orange`; unknown label returns itself; ordering |
| pooling | concatenation for `L`; link-sorted pool for `L_links` |
| Algorithm 2 | fruit beats city for `orange`; stability; unknown label leaves order unchanged |
| pipeline | $f_q$, $f_c$, $f_c+f_d$ on a three-entity fake graph; NIL for unknown labels |
| metrics | hit@k, Wilson interval, the paper's 2,328 |
| consistency | the offline analysis reproduces the stored experiment results label by label |

**Next:** Chapter 6 runs the experiment and compares with the published table.